In [1]:
import os
import shutil
import random
import yaml
import albumentations as A
import cv2

In [2]:
source_img = 'Dataset/Tyre_error_proofing.v15i.yolov9/train/images'
source_lbl = 'Dataset/Tyre_error_proofing.v15i.yolov9/train/labels'

target_image = 'Dataset/Tyre_error_proofing.v15i.yolov9/Augmented/images'
target_label = 'Dataset/Tyre_error_proofing.v15i.yolov9/Augmented/labels/'

os.makedirs(os.path.join(target_image), exist_ok=True)
os.makedirs(os.path.join(target_label), exist_ok=True)

id_to_name = {
    0: '80_label',
    1: 'black_wheel',
    2: 'simple_wheel',
    3: 'white_wheel'
}

# 3. Hisoblagichni shakllantirish
count_of_classes = {name: 0 for name in id_to_name.values()}

# 4. Sanash jarayoni
if not os.path.exists(source_lbl):
    print(f"Xato: {source_lbl} papkasi topilmadi!")
else:
    for filename in os.listdir(source_lbl):
        if filename.endswith('.txt'):
            with open(os.path.join(source_lbl, filename), 'r') as f:
                lines = f.readlines()
                for line in lines:
                    parts = line.split()
                    if len(parts) > 0:
                        class_id = int(parts[0])
                        if class_id in id_to_name:
                            class_name = id_to_name[class_id]
                            count_of_classes[class_name] += 1

    # 5. Natijani chiqarish
    print("--- Dataset statistikasi ---")
    for cls, count in count_of_classes.items():
        print(f"{cls:15}: {count} ta")

--- Dataset statistikasi ---
80_label       : 14644 ta
black_wheel    : 14434 ta
simple_wheel   : 14118 ta
white_wheel    : 14291 ta


In [25]:
transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),

    # Geometrik o'zgarishlar (Burchak cheklovlari bilan)
    A.OneOf([
        A.Affine(
            rotate=(-10, 10), 
            translate_percent={"x": (-0.05, 0.05), "y": (0.0, 0.30)}, 
            mode=0, cval=0, p=1.0
        ),
        A.Affine(
            rotate=(170, 190), 
            translate_percent={"x": (-0.05, 0.05), "y": (0.0, 0.30)}, 
            mode=0, cval=0, p=1.0
        ),
    ], p=1.0),
    
    A.MotionBlur(blur_limit=5, p=0.2), 
    
    A.GaussianBlur(blur_limit=3, p=0.2),

    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.Blur(blur_limit=3, p=0.1),

], 
bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']),
p=1.0,
is_check_shapes=True
)

C:\Users\Saidbek\AppData\Local\Temp\ipykernel_17012\897985990.py:7: UserWarning: Argument(s) 'mode, cval' are not valid for transform Affine
  A.Affine(
C:\Users\Saidbek\AppData\Local\Temp\ipykernel_17012\897985990.py:12: UserWarning: Argument(s) 'mode, cval' are not valid for transform Affine
  A.Affine(


In [26]:
def get_classes_from_file(label_path):
    with open(label_path, 'r') as f:
        return [int(line.split()[0]) for line in f.readlines()]

def move_files(img_name, folder):
    lbl_name = img_name.rsplit('.', 1)[0] + '.txt'
    shutil.move(os.path.join(source_img, img_name), os.path.join(target_image, img_name))
    shutil.move(os.path.join(source_lbl, lbl_name), os.path.join(target_label, lbl_name))
    moved_files.add(img_name)

def augment_dataset(input_img_dir, input_txt_dir, output_img_dir, output_txt_dir, transform_pipeline, num_variants):
    
    # Papkalarni yaratish
    os.makedirs(output_img_dir, exist_ok=True)
    os.makedirs(output_txt_dir, exist_ok=True)
    
    for file_img in os.listdir(input_img_dir):
        
        file_name = os.path.splitext(file_img)[0]
        
        img_path = os.path.join(input_img_dir, file_img)
        txt_path = os.path.join(input_txt_dir, file_name + ".txt")
        
        if os.path.exists(img_path) and os.path.exists(txt_path):            
            # Rasmni o'qish
            image = cv2.imread(img_path)
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            
            # Labelni o'qish (YOLO format: class x_center y_center width height)
            bboxes = []
            class_labels = []
            with open(txt_path, 'r') as f:
                for line in f:
                    parts = line.split()
                    if len(parts) == 5:
                        class_labels.append(int(parts[0]))
                        bboxes.append([float(x) for x in parts[1:]])
            
            # Augmentatsiya sikli
            for i in range(num_variants):
                try:
                    # Transformatsiyani qo'llash
                    transformed = transform_pipeline(image=image, bboxes=bboxes, class_labels=class_labels)
                    
                    new_name = f"{file_name}_v{i}"
                    
                    # Rasmni saqlash
                    save_path_img = os.path.join(output_img_dir, f"{new_name}.jpg")
                    cv2.imwrite(save_path_img, cv2.cvtColor(transformed['image'], cv2.COLOR_RGB2BGR))
                    
                    # Labelni saqlash
                    save_path_txt = os.path.join(output_txt_dir, f"{new_name}.txt")
                    with open(save_path_txt, 'w') as f:
                        for idx, box in enumerate(transformed['bboxes']):
                            f.write(f"{class_labels[idx]} {' '.join(map(str, box))}\n")
                            
                except Exception as e:
                    # Odatda bbox rasm tashqarisiga chiqib ketsa xato beradi
                    continue
        else:
            print(f"\nFayl topilmadi: {img_name} yoki {txt_name}")

    print(f"\nBajarildi! Jami yasalgan rasmlar: {len(os.listdir(output_img_dir))}")

def moving_files(initial_path,out_path):
    os.makedirs(out_path,exist_ok=True)
    for file in os.listdir(initial_path):
        shutil.move(os.path.join(initial_path,file),os.path.join(out_path,file))

In [27]:
# Faylning haqiqiy joylashgan yo'li
file_path = "Dataset/Tyre_error_proofing.v15i.yolov9/data.yaml"

with open(file_path, 'r') as file:
    data = yaml.safe_load(file)
    # print("Fayl muvaffaqiyatli o'qildi!")
    print(data["names"])

['80_label', 'black_wheel', 'simple_wheel', 'white_wheel']


In [29]:
augment_dataset(
    input_img_dir=source_img,
    input_txt_dir=source_lbl,
    output_img_dir=target_image,
    output_txt_dir=target_label,
    transform_pipeline=transform,
    num_variants=4
)


Bajarildi! Jami yasalgan rasmlar: 31928


In [30]:
# 1. Label fayllari joylashgan papka manzili

# 2. Sizning klasslaringiz tartibi (ID raqamlarini tekshirib oling!)
# data.yaml faylingizdagi tartib bilan bir xil bo'lishi shart

id_to_name = {
    0: '80_label',
    1: 'black_wheel',
    2: 'simple_wheel',
    3: 'white_wheel'
}

# 3. Hisoblagichni shakllantirish
count_of_classes = {name: 0 for name in id_to_name.values()}

# 4. Sanash jarayoni
if not os.path.exists(target_label):
    print(f"Xato: {target_label} papkasi topilmadi!")
else:
    for filename in os.listdir(target_label):
        if filename.endswith('.txt'):
            with open(os.path.join(target_label, filename), 'r') as f:
                lines = f.readlines()
                for line in lines:
                    parts = line.split()
                    if len(parts) > 0:
                        class_id = int(parts[0])
                        if class_id in id_to_name:
                            class_name = id_to_name[class_id]
                            count_of_classes[class_name] += 1

    # 5. Natijani chiqarish
    print("--- Dataset statistikasi ---")
    for cls, count in count_of_classes.items():
        print(f"{cls:15}: {count} ta")

--- Dataset statistikasi ---
80_label       : 11644 ta
black_wheel    : 11434 ta
simple_wheel   : 11119 ta
white_wheel    : 11291 ta


In [31]:
moving_files(target_image,source_img)

In [32]:
moving_files(target_label,source_lbl)

In [33]:
id_to_name = {
    0: '80_label',
    1: 'black_wheel',
    2: 'simple_wheel',
    3: 'white_wheel'
}

# 3. Hisoblagichni shakllantirish
count_of_classes = {name: 0 for name in id_to_name.values()}

# 4. Sanash jarayoni
if not os.path.exists(source_lbl):
    print(f"Xato: {source_lbl} papkasi topilmadi!")
else:
    for filename in os.listdir(source_lbl):
        if filename.endswith('.txt'):
            with open(os.path.join(source_lbl, filename), 'r') as f:
                lines = f.readlines()
                for line in lines:
                    parts = line.split()
                    if len(parts) > 0:
                        class_id = int(parts[0])
                        if class_id in id_to_name:
                            class_name = id_to_name[class_id]
                            count_of_classes[class_name] += 1

    # 5. Natijani chiqarish
    print("--- Dataset statistikasi ---")
    for cls, count in count_of_classes.items():
        print(f"{cls:15}: {count} ta")

--- Dataset statistikasi ---
80_label       : 14644 ta
black_wheel    : 14434 ta
simple_wheel   : 14118 ta
white_wheel    : 14291 ta
